# BME 603/606: Computational Methods for Biological Modeling & Simulation
## In-Class Activity: Balancing Chemical Reactions, Interpolation and Curve Fitting

1. **Balancing chemical reactions** — the stoichiometry of every reaction can be found by solving a linear system. We'll use the **nullspace** of an element matrix to extract the correct integer coefficients automatically.

2. **Polynomial interpolation & best fit** — given a handful of noisy measurements (think: a glucose sensor), we want a smooth curve that passes through (or near) every data point. We'll implement three different strategies — **Vandermonde**, **Lagrange**, and **Newton's divided differences**.

Let's get started!

## Part 1: Balancing Chemical Reactions

### Background: The Nullspace Method

Every balanced chemical equation is secretly a linear algebra problem.

**The recipe:**

1. **List every compound** (reactants and products).
2. **Build the element matrix A** — rows are chemical elements, columns are compounds.  
   Reactant columns get **positive** counts; product columns get **negative** counts.
3. **Find the nullspace** of A — the vector **x** such that A**x** = **0**.  
   Each entry of **x** is the stoichiometric coefficient of the corresponding compound.
4. **Scale to smallest positive integers**.

**Why does this work?** Conservation of mass requires that each element is balanced: the dot product of A with the coefficient vector must equal zero — exactly the definition of the nullspace!

```
Example walkthrough — combustion of methane:

  _ CH₄  +  _ O₂  →  _ CO₂  +  _ H₂O

       CH4   O2   CO2  H2O
  C  [  1    0    -1    0  ]
  H  [  4    0     0   -2  ]
  O  [  0    2    -2   -1  ]

  Nullspace vector → [1, 2, 1, 2]  =>  CH₄ + 2 O₂ → CO₂ + 2 H₂O
```

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.linalg import null_space
from IPython.display import HTML

# ── Utility: scale a nullspace vector to smallest positive integers ──────────
def to_integer_coeffs(v):
    """Given a real-valued nullspace vector, return positive integer coefficients."""
    v = v.flatten()
    # Make all signs consistent (first nonzero entry positive)
    nonzero = v[np.abs(v) > 1e-9]
    if len(nonzero) == 0:
        return v
    if nonzero[0] < 0:
        v = -v
    # Scale so the smallest nonzero magnitude is 1, then round
    v = v / np.min(np.abs(v[np.abs(v) > 1e-9]))
    return np.round(np.abs(v)).astype(int)

print("Imports done!")

Imports done!


### Exercise 1 (Guided): Glucose Combustion

Aerobic cellular respiration is the most important energy-releasing reaction in biology:

$$\_\, \text{C}_6\text{H}_{12}\text{O}_6 + \_\, \text{O}_2 \;\rightarrow\; \_\, \text{CO}_2 + \_\, \text{H}_2\text{O}$$

The element matrix A has **4 compounds** (columns) and **3 elements** (rows: C, H, O).  
Reactant coefficients are **positive**; product coefficients are **negative**.

Run the cell below to see the method in action, then use it as a template for the exercises that follow.

In [2]:
#       C6H12O6  O2   CO2  H2O
A = np.array([
    [  6,   0,  -1,   0],   # C
    [ 12,   0,   0,  -2],   # H
    [  6,   2,  -2,  -1],   # O
], dtype=float)

ns = null_space(A)          # columns span the nullspace
v  = ns[:, 0]               # take the first (and only) basis vector
coeffs = to_integer_coeffs(v)

compounds = ['C6H12O6', 'O2', 'CO2', 'H2O']
print("Balanced coefficients:")
for name, c in zip(compounds, coeffs):
    print(f"  {c}  {name}")
print()
print(f"  {coeffs[0]} C6H12O6  +  {coeffs[1]} O2  →  {coeffs[2]} CO2  +  {coeffs[3]} H2O")
print("  (Expected: 1 C6H12O6 + 6 O2 → 6 CO2 + 6 H2O)")

Balanced coefficients:
  1  C6H12O6
  6  O2
  6  CO2
  6  H2O

  1 C6H12O6  +  6 O2  →  6 CO2  +  6 H2O
  (Expected: 1 C6H12O6 + 6 O2 → 6 CO2 + 6 H2O)


### Exercise 2: Photosynthesis

Photosynthesis is the reverse of cellular respiration — plants use light energy to synthesize glucose from CO₂ and water:

$$\_\, \text{CO}_2 + \_\, \text{H}_2\text{O} \;\rightarrow\; \_\, \text{C}_6\text{H}_{12}\text{O}_6 + \_\, \text{O}_2$$

Build the element matrix and find the balanced coefficients.  

**Hint:** Compounds are CO₂, H₂O, C₆H₁₂O₆, O₂ (in that order). Reactants get + signs, products get − signs.

In [ ]:
#         CO2   H2O  C6H12O6   O2
A_photo = np.array([
    # C row:  Your values here
    # H row:  Your values here
    # O row:  Your values here
], dtype=float)

# Your code here — find the nullspace and print balanced coefficients
compounds_photo = ['CO2', 'H2O', 'C6H12O6', 'O2']

# Expected answer:  6 CO2  +  6 H2O  →  C6H12O6  +  6 O2

### Exercise 3: Ethanol Combustion

Fermentation produces ethanol (C₂H₅OH), which then burns in air:

$$\_\, \text{C}_2\text{H}_5\text{OH} + \_\, \text{O}_2 \;\rightarrow\; \_\, \text{CO}_2 + \_\, \text{H}_2\text{O}$$

Ethanol has the formula C₂H₆O (2 carbons, 6 hydrogens, 1 oxygen).  
Build the element matrix for the four elements involved (C, H, O) and balance the reaction.

In [ ]:
#           C2H5OH  O2   CO2  H2O
A_ethanol = np.array([
    # C row:
    # H row:
    # O row:
], dtype=float)

# Your code here
compounds_ethanol = ['C2H5OH', 'O2', 'CO2', 'H2O']

# Expected answer:  C2H5OH  +  3 O2  →  2 CO2  +  3 H2O

### Exercise 4: Ammonia Synthesis (Haber–Bosch Process)

The industrial synthesis of ammonia feeds roughly half the world's population by fixing atmospheric nitrogen into fertilizer:

$$\_\, \text{N}_2 + \_\, \text{H}_2 \;\rightarrow\; \_\, \text{NH}_3$$

This reaction involves only **2 elements** (N and H) and **3 compounds**.  
Build the 2×3 element matrix and find the balanced stoichiometry.

In [ ]:
#            N2   H2   NH3
A_ammonia = np.array([
    # N row:
    # H row:
], dtype=float)

# Your code here
compounds_ammonia = ['N2', 'H2', 'NH3']

# Expected answer:  N2  +  3 H2  →  2 NH3

### Exercise 5 (Optional): Build a General Chemical Balancer

Now generalise! Write a function `balance_reaction(A, compounds)` that:
1. Computes the nullspace of A
2. Extracts and scales integer coefficients
3. Returns a formatted string of the balanced equation, e.g.:
   `"1 CH4 + 2 O2 → 1 CO2 + 2 H2O"`

The function should split reactants (positive columns) from products (negative columns) using the sign convention from A's last column or by accepting a `n_reactants` argument.

Test your function on at least one of the reactions above and the **iron oxidation** reaction below:

$$\_\, \text{Fe} + \_\, \text{O}_2 \;\rightarrow\; \_\, \text{Fe}_2\text{O}_3$$

*Expected: 4 Fe + 3 O₂ → 2 Fe₂O₃*

In [ ]:
def balance_reaction(A, compounds, n_reactants):
    """
    Balance a chemical reaction.

    Parameters
    ----------
    A : np.ndarray, shape (n_elements, n_compounds)
        Element matrix. Reactant columns are positive, product columns negative.
    compounds : list of str
        Names of each compound (in column order of A).
    n_reactants : int
        Number of reactant compounds (first n_reactants columns).

    Returns
    -------
    str
        A formatted balanced equation string.
    """
    # Your code here
    pass

# Test: iron oxidation  Fe + O2 → Fe2O3
#        Fe   O2   Fe2O3
A_iron = np.array([
    [ 1,   0,  -2],   # Fe
    [ 0,   2,  -3],   # O
], dtype=float)

print(balance_reaction(A_iron, ['Fe', 'O2', 'Fe2O3'], n_reactants=2))
# Expected:  4 Fe + 3 O2 → 2 Fe2O3

# Also test one of your earlier reactions
# ...

---
## Part 2: Interpolation and Best Fit

### Background: The Interpolation Problem

A patient's blood glucose is measured at discrete time points.  
We want a smooth function $p(t)$ that **passes exactly through every measured value** so we can estimate glucose at unmeasured times.

| Time (hr) | Glucose (mg/dL) |
|:---------:|:---------------:|
| 0         | 90              |
| 2         | 115             |
| 4         | 140             |
| 6         | 100             |

Given $n+1$ distinct data points, there is a **unique polynomial of degree ≤ n** that passes through all of them.  

We'll find this polynomial three different ways:
1. **Vandermonde** — build and solve a linear system
2. **Lagrange** — explicit formula using basis functions
3. **Newton** — efficient recursive form for streaming data

All three give the **same** polynomial — they just differ in computation strategy.

In [3]:
# ── Shared data: blood glucose measurements ──────────────────────────────────
t_data = np.array([0., 2., 4., 6.])       # time points (hours)
y_data = np.array([90., 115., 140., 100.])  # glucose readings (mg/dL)

t_fine = np.linspace(-0.5, 6.5, 400)   # dense grid for plotting

def plot_interp(t_data, y_data, curves, labels, title):
    """Helper: scatter data and overlay interpolant curves."""
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(t_data, y_data, color='red', zorder=5, s=80, label='Data')
    for y_curve, label, style in curves:
        ax.plot(t_fine, y_curve, style, linewidth=2, label=label)
    ax.set_xlabel('Time (hr)')
    ax.set_ylabel('Glucose (mg/dL)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Exercise 7: Vandermonde Matrix Interpolation

For $n+1$ data points $(t_0, y_0), \ldots, (t_n, y_n)$, we fit the polynomial

$$p(t) = c_0 + c_1 t + c_2 t^2 + \cdots + c_n t^n$$

Plugging in each data point gives the linear system $\mathbf{A}\mathbf{c} = \mathbf{y}$, where **A** is the **Vandermonde matrix**:

$$A_{ij} = t_i^{\,j}, \quad i,j = 0, 1, \ldots, n$$

**Your tasks:**
1. Build the Vandermonde matrix **from scratch** using a loop (do not use `np.vander`).
2. Solve for the coefficients `c` using `np.linalg.solve`.
3. Evaluate $p(t)$ on `t_fine` and plot it against the data.
4. Estimate glucose at $t = 3$ hours.

In [ ]:
def build_vandermonde(x):
    """
    Build the Vandermonde matrix for nodes x.

    Parameters
    ----------
    x : 1-D array of length n+1

    Returns
    -------
    A : ndarray, shape (n+1, n+1)
        A[i, j] = x[i] ** j
    """
    n = len(x)
    A = np.zeros((n, n))
    # Hint: outer loop over rows i, inner loop over columns j
    # Your code here
    return A

# Build and solve
A_vdm = build_vandermonde(t_data)
print("Vandermonde matrix A:")
print(A_vdm)

c_vdm = None   # solve A_vdm @ c = y_data
print("\nCoefficients c:", c_vdm)

# Evaluate polynomial on t_fine
n = len(t_data)
p_vdm = None   # sum(c_vdm[j] * t_fine**j for j in range(n))

# Plot
# plot_interp(t_data, y_data, [(p_vdm, 'Vandermonde', 'b-')],
#             [], 'Vandermonde Interpolation')

# Estimate glucose at t = 3
t_query = 3.0
glucose_at_3 = None   # evaluate your polynomial at t_query
print(f"\nEstimated glucose at t={t_query} hr: {glucose_at_3:.2f} mg/dL")

### Exercise 8: Lagrange Interpolation

The **Lagrange basis function** $L_j(t)$ is 1 at $t_j$ and 0 at every other data point:

$$L_j(t) = \prod_{\substack{i=0 \\ i \neq j}}^{n} \frac{t - t_i}{t_j - t_i}$$

The interpolating polynomial is then simply the weighted sum:

$$p(t) = \sum_{j=0}^{n} y_j \, L_j(t)$$

**Advantages over Vandermonde:** No matrix solve needed! The $y_j$ values are directly the weights.

**Your tasks:**
1. Implement `lagrange_interp(t_data, y_data, t)` that evaluates $p(t)$ at any array of query points `t`.
2. Evaluate on `t_fine` and verify you get the same curve as Vandermonde.
3. Plot both methods on the same axes to confirm they agree.
4. Also plot each individual basis function $L_j(t)$ and verify the cardinal property.

In [ ]:
def lagrange_interp(t_data, y_data, t):
    """
    Evaluate the Lagrange interpolating polynomial at query points t.

    Parameters
    ----------
    t_data : 1-D array, length n+1   — interpolation nodes
    y_data : 1-D array, length n+1   — function values at nodes
    t      : scalar or 1-D array     — query points

    Returns
    -------
    p : same shape as t              — interpolated values
    """
    t = np.atleast_1d(np.asarray(t, dtype=float))
    n = len(t_data)
    p = np.zeros_like(t)
    for j in range(n):
        L_j = np.ones_like(t)
        for i in range(n):
            if i != j:
                pass   # multiply L_j by (t - t_data[i]) / (t_data[j] - t_data[i])
        pass   # add y_data[j] * L_j to p
    return p

p_lag = lagrange_interp(t_data, y_data, t_fine)

# 1. Compare Vandermonde vs Lagrange
# plot_interp(t_data, y_data,
#             [(p_vdm, 'Vandermonde', 'b-'), (p_lag, 'Lagrange', 'r--')],
#             [], 'Vandermonde vs Lagrange (should overlap exactly)')

# 2. Plot all four Lagrange basis functions on [-0.5, 6.5]
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharey=False)
for j in range(4):
    ax = axes[j]
    # Compute L_j(t_fine) — replace the pass below
    L_j = np.ones_like(t_fine)
    for i in range(4):
        if i != j:
            pass   # L_j *= ...
    ax.plot(t_fine, L_j, 'b-', linewidth=2)
    ax.scatter(t_data, [1 if i == j else 0 for i in range(4)],
               color='red', zorder=5)
    ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
    ax.set_title(f'$L_{j}(t)$')
    ax.set_xlabel('t (hr)')
plt.suptitle('Lagrange Basis Functions', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Glucose estimate at t=3 hr: {lagrange_interp(t_data, y_data, 3.0)[0]:.2f} mg/dL")

### Exercise 9: Newton's Divided Differences

Newton's form expresses the interpolating polynomial using a different basis:

$$p(t) = c_0 + c_1(t-t_0) + c_2(t-t_0)(t-t_1) + \cdots + c_n(t-t_0)\cdots(t-t_{n-1})$$

The coefficients $c_k$ are **divided differences**, computed by the recursive table:

| $c_0 = y_0$ | $c_1 = \dfrac{y_1-y_0}{t_1-t_0}$ | $c_2 = \dfrac{f[t_1,t_2]-f[t_0,t_1]}{t_2-t_0}$ | $\cdots$ |

**Key advantage:** When a **new data point arrives**, only ONE new coefficient is needed — all previous coefficients remain unchanged!

**Your tasks:**
1. Implement `divided_differences(t, y)` that returns the coefficient array `c`.
2. Implement `newton_interp(t_data, c, t)` that evaluates the Newton form using **Horner's method** (nested multiplication).
3. Verify the result matches Vandermonde and Lagrange.
4. Demonstrate the update property: add a new data point at $t=8$, glucose = 82 mg/dL, and show that only the last coefficient changes.

In [ ]:
def divided_differences(t, y):
    """
    Compute Newton divided difference coefficients.

    Uses the in-place upper-triangular algorithm:
      Start with c = y.copy()
      For k = 1, ..., n-1:
        c[k:] = (c[k:] - c[k-1:-1 or k-1]) / (t[k:] - t[:n-k])

    Parameters
    ----------
    t : 1-D array of nodes
    y : 1-D array of function values

    Returns
    -------
    c : 1-D array — divided difference coefficients c[0], c[1], ..., c[n-1]
    """
    n = len(t)
    c = y.copy().astype(float)
    for k in range(1, n):
        pass   # update c[k:] in place
    return c

def newton_interp(t_data, c, t):
    """
    Evaluate the Newton interpolating polynomial via Horner's method.

    p(t) = c[n-1]
    for k = n-2 downto 0:
        p(t) = p(t) * (t - t_data[k]) + c[k]

    Parameters
    ----------
    t_data : nodes used to build c (length n)
    c      : divided difference coefficients (length n)
    t      : query points (scalar or array)

    Returns
    -------
    Evaluated polynomial values
    """
    t = np.atleast_1d(np.asarray(t, dtype=float))
    result = np.full_like(t, c[-1])
    for k in range(len(c) - 2, -1, -1):
        pass   # result = result * (t - t_data[k]) + c[k]
    return result

# Compute coefficients and evaluate
c_newton = divided_differences(t_data, y_data)
print("Newton divided difference coefficients:", c_newton)

p_newton = newton_interp(t_data, c_newton, t_fine)

# Compare all three methods
# plot_interp(t_data, y_data,
#             [(p_vdm,    'Vandermonde', 'b-'),
#              (p_lag,    'Lagrange',    'r--'),
#              (p_newton, 'Newton',      'g:')],
#             [], 'All Three Methods (should be identical)')

# ── Demonstrate the update property ─────────────────────────────────────────
t_new = np.append(t_data, 8.0)
y_new = np.append(y_data, 82.0)

c_updated = divided_differences(t_new, y_new)
print("\nOriginal coefficients:", c_newton)
print("Updated  coefficients:", c_updated)
print("\nAre the first 4 coefficients the same?")
print(np.allclose(c_newton, c_updated[:4]))

### Exercise 10: Least Squares Best Fit for Noisy Data

Real glucose sensors are noisy. Interpolation would pass through every noisy point — that's **overfitting**.

Instead, we find the polynomial that **minimises the sum of squared residuals** over many more data points. For a degree-$d$ polynomial with $m \gg d+1$ data points, the overdetermined system $\mathbf{A}\mathbf{c} \approx \mathbf{y}$ (where A is the tall Vandermonde matrix, shape $m \times (d+1)$) has the **least-squares solution**:

$$\mathbf{c}^* = (\mathbf{A}^\top \mathbf{A})^{-1} \mathbf{A}^\top \mathbf{y}$$

**Your tasks:**
1. Generate 30 noisy glucose readings over $t \in [0, 8]$ hr using the true trend plus Gaussian noise.
2. Fit polynomials of degree 1, 2, and 4 using the **normal equations** (implement matrix form, not `np.polyfit`).
3. Also solve using `np.linalg.lstsq` and confirm you get the same answer.
4. Plot all fits on the same axes. Which degree captures the trend without overfitting?

In [ ]:
rng = np.random.default_rng(42)
t_noisy = np.linspace(0, 8, 30)                        # 30 hourly readings
y_true   = 90 + 20*t_noisy - 3*t_noisy**2             # true quadratic trend
y_noisy  = y_true + rng.normal(0, 8, size=30)          # add noise (σ = 8 mg/dL)

def least_squares_poly(t, y, degree):
    """
    Fit a polynomial of the given degree to (t, y) via the normal equations.

    Returns coefficients c such that p(t) = c[0] + c[1]*t + ... + c[d]*t^d.
    """
    # Build the (m x d+1) design matrix — each column is t**j
    # Your code here
    A = None   # shape (len(t), degree+1)

    # Solve normal equations: (A^T A) c = A^T y
    c = None   # np.linalg.solve(A.T @ A, A.T @ y)
    return c

# Fit three polynomial degrees
t_plot = np.linspace(0, 8, 300)
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(t_noisy, y_noisy, alpha=0.6, color='gray', label='Noisy readings', zorder=3)
ax.plot(t_plot, 90 + 20*t_plot - 3*t_plot**2, 'k--', linewidth=1.5, label='True trend')

colors = ['steelblue', 'tomato', 'seagreen']
for deg, col in zip([1, 2, 4], colors):
    c = least_squares_poly(t_noisy, y_noisy, deg)
    if c is not None:
        p_fit = sum(c[j] * t_plot**j for j in range(deg+1))
        ax.plot(t_plot, p_fit, color=col, linewidth=2, label=f'Degree {deg}')

ax.set_xlabel('Time (hr)')
ax.set_ylabel('Glucose (mg/dL)')
ax.set_title('Least-Squares Polynomial Fits')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verify using np.linalg.lstsq
A2 = np.column_stack([t_noisy**j for j in range(3)])   # degree-2 design matrix
c_lstsq, _, _, _ = np.linalg.lstsq(A2, y_noisy, rcond=None)
c_normal = least_squares_poly(t_noisy, y_noisy, 2)
if c_normal is not None:
    print("lstsq coefficients:  ", np.round(c_lstsq, 4))
    print("normal eqn coeff:    ", np.round(c_normal, 4))
    print("Match?", np.allclose(c_lstsq, c_normal, atol=1e-6))

---
## Challenge Exercise (Optional): Live Newton Interpolant — Streaming Glucose Monitor

### Background

A **continuous glucose monitor (CGM)** transmits a new blood glucose reading every hour to an algorithm running on a smartphone. The algorithm needs to maintain an up-to-date interpolating polynomial and plot the estimated glucose curve in real time.

In this challenge you will:
1. Simulate a CGM sensor sending readings one at a time.
2. After each new reading, update the Newton coefficient array using only the new divided difference.
3. Create an **animated plot** that shows the polynomial evolving as data accumulates — exactly what a real CGM algorithm would display.

**Key things to implement:**  
When point $(t_k, y_k)$ arrives, the new top-level divided difference is:
$$c_k = \frac{f[t_1, \ldots, t_k] - f[t_0, \ldots, t_{k-1}]}{t_k - t_0}$$
You can extract this as `divided_differences(t_so_far, y_so_far)[-1]` — only the last entry is new.

In [ ]:
# ── 1. Simulate CGM data (12 hours, one reading per hour) ───────────────────
rng_cgm = np.random.default_rng(7)
t_cgm = np.arange(0, 12, dtype=float)                       # t = 0,1,...,11 hr
# True glucose: peaks after a meal, then falls
y_true_cgm = 95 + 30*np.exp(-((t_cgm - 3)**2)/4) - 4*(t_cgm - 6) + 5*np.sin(t_cgm)
y_cgm = y_true_cgm + rng_cgm.normal(0, 4, size=len(t_cgm))  # add mild sensor noise

print("Simulated CGM readings (time, glucose):")
for ti, yi in zip(t_cgm, y_cgm):
    print(f"  t={ti:.0f} hr  →  {yi:.1f} mg/dL")

In [ ]:
# ── 2. Efficient online Newton update ───────────────────────────────────────

def newton_update(t_prev, c_prev, t_new, y_new):
    """
    Add one new data point to an existing Newton interpolant.

    Parameters
    ----------
    t_prev : 1-D array of previous nodes (length k)
    c_prev : 1-D array of existing divided difference coefficients (length k)
    t_new  : float — the new node
    y_new  : float — the new function value

    Returns
    -------
    t_all  : updated node array (length k+1)
    c_all  : updated coefficient array (length k+1)
    """
    t_all = np.append(t_prev, t_new)
    y_all = np.array([newton_interp(t_prev, c_prev, ti) for ti in t_all[:-1]])
    # The new node evaluates exactly to y_new; previous nodes evaluate via the
    # existing polynomial.  We only need the TOP-LEVEL divided difference:
    y_all = np.append(y_all, y_new)
    # Reuse divided_differences but extract only the last (new) coefficient
    # Hint: you can call divided_differences on the full (t_all, y_cgm[:len(t_all)])
    # OR compute the new c_k incrementally.  Either approach is fine.
    c_all = divided_differences(t_all, y_cgm[:len(t_all)])
    return t_all, c_all

# ── 3. Collect frames for animation ─────────────────────────────────────────
t_plot_cgm = np.linspace(-0.2, 11.2, 500)
frames_data   = []   # (t_so_far, y_so_far, t_plot, p_plot) for each step

# Your code here:
# Start with the first two data points (need ≥2 for degree-1 poly)
# Then loop through t_cgm[2:] and update using newton_update or divided_differences
# After each update, evaluate the Newton polynomial on t_plot_cgm and store the frame

# Starter:
for k in range(2, len(t_cgm) + 1):
    t_so_far = t_cgm[:k]
    y_so_far = y_cgm[:k]
    c_so_far = divided_differences(t_so_far, y_so_far)
    p_plot   = newton_interp(t_so_far, c_so_far, t_plot_cgm)
    frames_data.append((t_so_far.copy(), y_so_far.copy(), p_plot.copy()))

print(f"Built {len(frames_data)} animation frames.")

In [ ]:
# ── 4. Animate! ──────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t_cgm, y_true_cgm, 'k--', linewidth=1.2, alpha=0.5, label='True glucose')
ax.set_xlim(-0.3, 11.3)
ax.set_ylim(60, 180)
ax.set_xlabel('Time (hr)')
ax.set_ylabel('Glucose (mg/dL)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

scatter = ax.scatter([], [], color='red', zorder=5, s=70, label='CGM readings')
line,   = ax.plot([], [], 'b-', linewidth=2.5, label='Newton interpolant')
title   = ax.set_title('CGM Reading: t = 0 hr', fontsize=13)

def animate(i):
    t_so_far, y_so_far, p_plot = frames_data[i]
    scatter.set_offsets(np.column_stack([t_so_far, y_so_far]))
    # Only plot the interpolant within the range of observed data
    mask = (t_plot_cgm >= t_so_far[0]) & (t_plot_cgm <= t_so_far[-1])
    line.set_xdata(t_plot_cgm[mask])
    line.set_ydata(p_plot[mask])
    k = len(t_so_far)
    title.set_text(f'CGM stream: {k} readings  |  Newton degree {k-1}  |  t = {t_so_far[-1]:.0f} hr')
    return scatter, line, title

anim = animation.FuncAnimation(
    fig, animate, frames=len(frames_data), interval=700, blit=True
)
plt.close(fig)
HTML(anim.to_jshtml())

### Challenge Extension: Runge's Phenomenon

High-degree polynomial interpolation is not always a good idea!

**Runge's phenomenon:** As you add more equally-spaced nodes, the interpolating polynomial can oscillate wildly near the edges of the interval — even when the underlying function is smooth.

**Your task:**
1. Choose the function $f(x) = \dfrac{1}{1 + 25x^2}$ on $[-1, 1]$ (Runge's classic example).
2. Interpolate using $n+1$ **equally-spaced** nodes for $n = 4, 8, 12, 16$.
3. Plot all four interpolants alongside the true function.
4. Now repeat using **Chebyshev nodes**: $x_k = \cos\!\left(\dfrac{(2k+1)\pi}{2(n+1)}\right)$, $k = 0,\ldots,n$.
5. Compare: do Chebyshev nodes cure the oscillations? Why?

*Hint: use your `lagrange_interp` or `divided_differences` / `newton_interp` functions — they work for any set of nodes.*

In [ ]:
def runge(x):
    return 1.0 / (1 + 25 * x**2)

x_dense = np.linspace(-1, 1, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
degrees = [4, 8, 12, 16]
colors  = ['steelblue', 'tomato', 'seagreen', 'darkorchid']

for ax, node_type in zip(axes, ['Equally spaced', 'Chebyshev nodes']):
    ax.plot(x_dense, runge(x_dense), 'k-', linewidth=2, label='True f(x)')
    for n, col in zip(degrees, colors):
        if node_type == 'Equally spaced':
            x_nodes = np.linspace(-1, 1, n + 1)
        else:
            k = np.arange(n + 1)
            x_nodes = np.cos((2*k + 1) * np.pi / (2*(n + 1)))
            x_nodes = np.sort(x_nodes)
        y_nodes = runge(x_nodes)
        # Interpolate using your Newton or Lagrange implementation
        # p_interp = ...
        # ax.plot(x_dense, p_interp, color=col, linewidth=1.5, label=f'n={n}')
        pass
    ax.set_title(node_type, fontsize=12)
    ax.set_xlabel('x')
    ax.set_ylim(-0.5, 1.5)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('f(x)')
plt.suptitle("Runge's Phenomenon", fontsize=14)
plt.tight_layout()
plt.show()

---
## Further Exploration

### Chemical Reaction Networks

The nullspace method extends beyond simple reactions. In a full **stoichiometric matrix** (columns = reactions, rows = species), the right nullspace gives **steady-state flux distributions** — the basis of flux balance analysis (FBA) used in metabolic engineering and drug discovery.

### Interpolation in Higher Dimensions

Everything we did today was 1D (time). In imaging and spatial biology you need **2D or 3D interpolation**: bilinear interpolation for images, radial basis functions (RBFs) for scattered spatial data, and splines for smooth surface fitting.

### Splines — the Practical Choice

High-degree global polynomials suffer from Runge's phenomenon (above). In practice, **cubic splines** are used: piecewise degree-3 polynomials stitched together with smooth joins. `scipy.interpolate.CubicSpline` implements this. Splines underlie the smooth curves in genomic browsers, medical imaging reconstruction, and finite element methods.

### References & Resources

- [Stoichiometry & linear algebra — brilliant.org](https://brilliant.org/wiki/stoichiometry/)
- [Flux Balance Analysis — Wikipedia](https://en.wikipedia.org/wiki/Flux_balance_analysis)
- [NumPy `null_space` documentation](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.null_space.html)
- [Polynomial Interpolation — Wikipedia](https://en.wikipedia.org/wiki/Polynomial_interpolation)
- [Runge's Phenomenon — Wikipedia](https://en.wikipedia.org/wiki/Runge%27s_phenomenon)
- [SciPy Interpolation Tutorial](https://docs.scipy.org/doc/scipy/tutorial/interpolate.html)

---